In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual")
metadata_dir = save_dir / "metadata"

gauges = gpd.read_parquet(metadata_dir / "gauges.parquet").set_index('site_id')

subs_df = pd.read_parquet(metadata_dir / 'subbasins.parquet', columns=['site_id', 'is_gauge', 'uparea_km2', 'outlet_id'])
subs_df.index = subs_df.index.astype(str)

basin_subbasin_dict = subs_df.groupby('outlet_id').apply(lambda g: g.index.tolist(), include_groups=False).to_dict()

In [2]:
from data import ZarrBasinStore
store_path = Path("/scratch4/workspace/tlanghorst_umass_edu-zbs_vD")
store = ZarrBasinStore(store_path)

store.compute_and_store_stats()

Dask Dashboard: http://127.0.0.1:8787/status


Processing Basins: 0it [00:00, ?it/s]


In [3]:
subs_df['original_comid'] = subs_df.index

# A small % of gauges are dropped if they don't match well.
common_idx = subs_df.index.intersection(gauges.index)
subs_df.loc[common_idx, 'original_comid'] = gauges.loc[common_idx, 'COMID']

subs_df

,is_gauge,uparea_km2,outlet_id,original_comid
site_id,,,,
23021321,False,209.123678,EAUF-A3750050,23021321
23021261,False,213.109547,EAUF-A3750050,23021261
23021244,False,291.030672,EAUF-A3750050,23021244
23021122,False,477.964586,EAUF-A3750050,23021122
23021112,False,394.295017,EAUF-A3750050,23021112
...,...,...,...,...
81003765,False,1613.152478,USGS-15908000,81003765
81001414,False,1473.949940,USGS-15908000,81001414
81001413,False,2751.705206,USGS-15908000,81001413


In [ ]:
twsa_df.index.isin(subs_df['original_comid']

In [ ]:
twsa_df = pd.read_csv(region_file, index_col='COMID')
twsa_df.index = twsa_df.index.astype(int).astype(str)
twsa_df = twsa_df.drop(columns=['Unnamed: 0'])

base_date = pd.Timestamp('2002-01-01')

# convert column names to datetime
twsa_df.columns = [
    (base_date + pd.to_timedelta(float(col), unit='D')).date()
    for col in twsa_df.columns
]

twsa_df

In [ ]:
subs_df['original_comid'].unique()

In [ ]:
twsa_long = (
    twsa_df
    .stack()
    .rename('twsa')
)
twsa_long.index.names = ['COMID','datetime']

twsa_long

In [ ]:
datasets = Path("/nas/cee-ice/data")
base_date = pd.Timestamp('2002-01-01')

comids = subs_df['original_comid'].astype(str).unique()
pfaf2s = (subs_df['original_comid'].astype(int)//1E6).astype(int).astype(str).unique()

twsa_dfs = []
for pfaf2 in tqdm(pfaf2s):
    twsa_file = datasets / 'GRACE_TWSA' / f'region_{pfaf2[0]}' / f'twsa_{pfaf2}.csv'
    
    twsa_df = pd.read_csv(twsa_file, index_col='COMID')
    twsa_df.index = twsa_df.index.astype(int).astype(str)
    twsa_df = twsa_df.drop(columns=['Unnamed: 0'])
    
    # convert column names to datetime
    twsa_df.columns = [
        (base_date + pd.to_timedelta(float(col), unit='D')).date()
        for col in twsa_df.columns
    ]

    # Drop unused comids
    twsa_df = twsa_df[twsa_df.index.isin(comids)]

    twsa_long = (
        twsa_df
        .stack()
        .rename('twsa')
    )
    twsa_long.index.names = ['COMID','datetime']

    twsa_dfs.append(twsa_long)
                    
twsa = pd.concat(twsa_dfs)

# twsa

In [ ]:
twsa

In [ ]:
twsa.xs('23000929', level='COMID').plot()

In [ ]:

def get_grace_twsa(comid):
    try:
        twsa_ix = twsa.xs(comid, level='COMID')
    except KeyError:
        return None
    
    twsa_ix.index = pd.to_datetime(twsa_ix.index).tz_localize('UTC')
    twsa_ix.index.name = 'date'
    
    return twsa_ix.to_frame().rename(columns={'twsa':'grace_twsa'})


BATCH_SIZE = 1000
batches = []
for basin_id, basin_group in subs_df.groupby('outlet_id'):
    basin_subs = basin_subbasin_dict[basin_id]
    n_subs = len(basin_subs)

    for i in range(0, n_subs, BATCH_SIZE):
        batch_subs = basin_subs[i : i + BATCH_SIZE]
        batches.append((basin_id, batch_subs))

        
# Iterate over basins (distinct Zarr stores)
for basin_id, batch_subs in tqdm(batches):
    basin_subs = basin_subbasin_dict[basin_id]
    batch_df_list = []
    for subbasin in batch_subs:
        comid = subs_df.loc[subbasin]['original_comid']

        site_df = get_grace_twsa(comid)
            
        if site_df is not None and not site_df.empty:
            site_df['subbasin'] = subbasin
            batch_df_list.append(site_df)

    # If the whole batch is empty, we technically don't need to write anything 
    # since the zarr was initialized with NaNs/empty).
    if batch_df_list:
        batch_df = pd.concat(batch_df_list)
        store.write_batch_data(basin_id, batch_df, basin_subs, batch_subs)

In [ ]:
basin_paths = [p for p in store.store_path.iterdir() if p.is_dir()]

basin_paths[0]

In [ ]:
store.compute_and_store_stats()